# Phase 3: Data Mining and Feature Engineering

This notebook covers the transformation of raw financial data into predictive features.

## 1. Imports and Reproducibility

In [1]:
import pandas as pd
import numpy as np
import os
from statsmodels.tsa.stattools import adfuller
import warnings
warnings.filterwarnings('ignore')

SEED = 25
np.random.seed(SEED)

TICKERS = ["AAPL", "MSFT", "JPM", "SPY", "GLD", "VIX", "TNX", "DX", "XLK", "XLF"]
CORE_TICKERS = ["AAPL", "MSFT", "JPM", "SPY", "GLD"]
DAILY_DIR = "../data/raw/daily"
HOURLY_DIR = "../data/raw/hourly"
PROCESSED_DIR = "../data/features"
os.makedirs(PROCESSED_DIR, exist_ok=True)

print(f"Reproducibility SEED: {SEED}")

Reproducibility SEED: 25


## 2. Technical Indicators & FFD Engine

In [2]:
def apply_ffd(series, d, threshold=1e-4):
    w = [1.0]
    for k in range(1, len(series)):
        w_ = -w[-1] / k * (d - k + 1)
        if abs(w_) < threshold: break
        w.append(w_)
    w = np.array(w[::-1]).reshape(-1, 1)
    width = len(w) - 1
    res = []
    series_array = series.values
    for i in range(width, len(series)):
        res.append(np.dot(w.T, series_array[i-width:i+1])[0])
    return pd.Series(res, index=series.index[width:])

def find_min_d(series):
    if len(series) < 50: return 1.0
    for d in np.linspace(0, 1, 6):
        ffd_series = apply_ffd(series, d)
        if len(ffd_series) < 20: continue
        try:
            if adfuller(ffd_series.dropna())[1] < 0.05: return d
        except: continue
    return 1.0

def add_technical_indicators(df):
    df['RSI'] = 100 - (100 / (1 + (df['Close'].diff().where(df['Close'].diff() > 0, 0).rolling(14).mean() / 
                                  -df['Close'].diff().where(df['Close'].diff() < 0, 0).rolling(14).mean())))
    df['MACD'] = df['Close'].ewm(span=12).mean() - df['Close'].ewm(span=26).mean()
    df['Vol'] = df['Close'].pct_change().rolling(20).std()
    return df

def apply_triple_barrier(df, horizon=5):
    # Label 1 if any price in next 'horizon' spans is > boundary, -1 if < boundary, else 0
    vol = df['Close'].pct_change().rolling(20).std() * np.sqrt(horizon)
    labels = []
    for i in range(len(df)):
        if i + horizon >= len(df) or pd.isna(vol.iloc[i]):
            labels.append(np.nan)
            continue
        p0 = df['Close'].iloc[i]
        ub, lb = p0 * (1 + vol.iloc[i]), p0 * (1 - vol.iloc[i])
        window = df['Close'].iloc[i+1 : i+horizon+1]
        if any(window >= ub): labels.append(1)
        elif any(window <= lb): labels.append(-1)
        else: labels.append(0)
    return pd.Series(labels, index=df.index)

## 3. Processing Logic (Soft Join)

In [3]:
def generate_feature_set(horizon="daily"):
    data_dir = DAILY_DIR if horizon == "daily" else HOURLY_DIR
    processed_dfs = {}
    
    for ticker in TICKERS:
        print(f"Processing {ticker} ({horizon})...")
        path = os.path.join(data_dir, f"{ticker}.csv")
        df = pd.read_csv(path, skiprows=[1,2], index_col=0, parse_dates=True)
        df.index = pd.to_datetime(df.index, utc=True).tz_localize(None)
        df = df.apply(pd.to_numeric, errors='coerce').dropna()
        
        # Basic indicators
        df = add_technical_indicators(df)
        
        # FFD
        d = find_min_d(np.log(df['Close']))
        df['FFD'] = apply_ffd(np.log(df['Close']), d)
        
        # Labels for core
        if ticker in CORE_TICKERS:
            df['Target'] = apply_triple_barrier(df, horizon=5 if horizon=="daily" else 8)
            
        df.columns = [f"{ticker}_{c}" for c in df.columns]
        processed_dfs[ticker] = df

    # Assemble master
    anchor = processed_dfs['SPY'].index
    master = pd.DataFrame(index=anchor)
    
    for ticker in TICKERS:
        t_df = processed_dfs[ticker]
        # Resynchronize if necessary (especially for hourly indices)
        if horizon == "hourly":
            t_df = t_df.reindex(anchor, method='ffill', limit=1)
        master = master.join(t_df, how='left')
    
    # SOFT CLEANING
    # 1. Forward fill feature gaps
    master = master.ffill()
    
    # 2. DROP ONLY IF Target Assets (Core) are missing or Fundamental data is NaN at very start
    # We focus on JPM as a proxy for the core signal
    master = master.dropna(subset=['JPM_Target', 'AAPL_Target'])
    master = master.dropna() # Final fallback for remaining NaNs
    
    save_path = os.path.join(PROCESSED_DIR, f"{horizon}_features.csv")
    master.to_csv(save_path)
    print(f"Saved {horizon} features. Final Shape: {master.shape}")
    return master

print("Execution started...")
daily_m = generate_feature_set("daily")
hourly_m = generate_feature_set("hourly")

Execution started...
Processing AAPL (daily)...


Processing MSFT (daily)...


Processing JPM (daily)...


Processing SPY (daily)...


Processing GLD (daily)...


...

Processing VIX (hourly)...


Processing TNX (hourly)...
Processing DX (hourly)...


Processing XLK (hourly)...


Processing XLF (hourly)...


Saved hourly features. Final Shape: (2963, 95)
